In [1]:
#Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
DATA = Path.cwd().parent / "data"

#Import data
filedata = DATA / "ecommerce_raw.csv"
readexcel = pd.read_csv(filedata)
df= pd.DataFrame(readexcel)
print(df)

     Customer_ID  Age  Gender Income_Level Marital_Status Education_Level  \
0    37-611-6911   22  Female       Middle        Married      Bachelor's   
1    29-392-9296   49    Male         High        Married     High School   
2    84-649-5117   24  Female       Middle         Single        Master's   
3    48-980-6078   29  Female       Middle         Single        Master's   
4    91-170-9072   33  Female       Middle        Widowed     High School   
..           ...  ...     ...          ...            ...             ...   
995  20-562-2569   44  Female         High       Divorced      Bachelor's   
996  41-366-4205   50  Female         High         Single     High School   
997  77-241-7621   26    Male         High        Married      Bachelor's   
998  53-091-2176   21  Female         High        Widowed      Bachelor's   
999  48-203-9118   21  Female       Middle         Single     High School   

    Occupation    Location     Purchase_Category Purchase_Amount  ...  \
0 

In [3]:
#Spesifikasi data
print(f"Shape awal  : {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Kolom       : {df.columns.tolist()}")

Shape awal  : 1000 baris, 28 kolom
Kolom       : ['Customer_ID', 'Age', 'Gender', 'Income_Level', 'Marital_Status', 'Education_Level', 'Occupation', 'Location', 'Purchase_Category', 'Purchase_Amount', 'Frequency_of_Purchase', 'Purchase_Channel', 'Brand_Loyalty', 'Product_Rating', 'Time_Spent_on_Product_Research(hours)', 'Social_Media_Influence', 'Discount_Sensitivity', 'Return_Rate', 'Customer_Satisfaction', 'Engagement_with_Ads', 'Device_Used_for_Shopping', 'Payment_Method', 'Time_of_Purchase', 'Discount_Used', 'Customer_Loyalty_Program_Member', 'Purchase_Intent', 'Shipping_Preference', 'Time_to_Decision']


In [4]:
#Kondisi awal data
print("\n[2.1] Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n[2.2] Duplicate Rows  : {df.duplicated().sum()}")
print(f"[2.3] Tipe data Purchase_Amount : {df['Purchase_Amount'].dtype}")
print(f"[2.4] Sample Purchase_Amount    : {df['Purchase_Amount'].head(3).tolist()}")
print(f"\n[2.5] Nilai unik Gender:\n{df['Gender'].value_counts()}")
print(f"\n[2.6] Nilai unik Occupation:\n{df['Occupation'].value_counts()}")


[2.1] Missing Values:
Social_Media_Influence    247
Engagement_with_Ads       256
dtype: int64

[2.2] Duplicate Rows  : 0
[2.3] Tipe data Purchase_Amount : object
[2.4] Sample Purchase_Amount    : ['$333.80 ', '$222.22 ', '$426.22 ']

[2.5] Nilai unik Gender:
Gender
Female         452
Male           449
Bigender        20
Agender         19
Genderfluid     17
Non-binary      16
Polygender      15
Genderqueer     12
Name: count, dtype: int64

[2.6] Nilai unik Occupation:
Occupation
High      517
Middle    483
Name: count, dtype: int64


In [5]:
#Hapus duplikat
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Baris sebelum : {before}")
print(f"Baris sesudah : {after}")
print(f"Duplikat dihapus : {before - after}")

Baris sebelum : 1000
Baris sesudah : 1000
Duplikat dihapus : 0


In [6]:
#Hapus Income_Level
print(f"Jumlah kolom sebelum drop: {df.shape[1]}")
df = df.drop(columns=['Income_Level'])
print(f"Jumlah kolom setelah drop: {df.shape[1]}")
print("'Income_Level' berhasil dihapus.")

Jumlah kolom sebelum drop: 28
Jumlah kolom setelah drop: 27
'Income_Level' berhasil dihapus.


In [7]:
#Ubah format purchase amount dari $ menjadi float
df["Purchase_Amount"] = (
    df["Purchase_Amount"]
    .str.replace("$", "", regex=False)
    .str.strip()
    .astype(float)
)
print(f"Tipe data sekarang : {df['Purchase_Amount'].dtype}")
print(f"Min : {df['Purchase_Amount'].min()}")
print(f"Max : {df['Purchase_Amount'].max()}")
print(f"Mean: {df['Purchase_Amount'].mean():.2f}")

Tipe data sekarang : float64
Min : 50.71
Max : 498.33
Mean: 275.06


In [8]:
#Konversi time of purchase menjadi format datetime
df["Time_of_Purchase"] = pd.to_datetime(df["Time_of_Purchase"])
df["Purchase_Month"]     = df["Time_of_Purchase"].dt.month
df["Purchase_Month_Name"]= df["Time_of_Purchase"].dt.strftime("%b")
df["Purchase_Year"]      = df["Time_of_Purchase"].dt.year
df["Purchase_DayOfWeek"] = df["Time_of_Purchase"].dt.day_name()
df["Purchase_Quarter"]   = df["Time_of_Purchase"].dt.quarter
print(f"Tipe data sekarang : {df['Time_of_Purchase'].dtype}")
print(f"Rentang tanggal    : {df['Time_of_Purchase'].min().date()} → {df['Time_of_Purchase'].max().date()}")
print(f"Kolom baru         : Purchase_Month, Purchase_Month_Name, Purchase_Year, Purchase_DayOfWeek, Purchase_Quarter")

Tipe data sekarang : datetime64[ns]
Rentang tanggal    : 2024-01-01 → 2024-12-30
Kolom baru         : Purchase_Month, Purchase_Month_Name, Purchase_Year, Purchase_DayOfWeek, Purchase_Quarter


In [9]:
#Standarisasi kolom gender
standard_gender = ["Male", "Female"]
df["Gender"] = df["Gender"].apply(
    lambda x: x if x in standard_gender else "Other"
)
print("Distribusi Gender setelah standarisasi:")
print(df["Gender"].value_counts())

Distribusi Gender setelah standarisasi:
Gender
Female    452
Male      449
Other      99
Name: count, dtype: int64


In [10]:
#Ubah collumn occupation menjadi income class (high/middle)
df.rename(columns={"Occupation": "Income_Class"}, inplace=True)
print("Kolom 'Occupation' berhasil di-rename menjadi 'Income_Class'.")
print(f"Jumlah kolom sekarang: {df.shape[1]}")
print(f"Nilai unik Income_Class: {df['Income_Class'].value_counts().to_dict()}")

Kolom 'Occupation' berhasil di-rename menjadi 'Income_Class'.
Jumlah kolom sekarang: 32
Nilai unik Income_Class: {'High': 517, 'Middle': 483}


In [11]:
#Menangani missing value collumn social media influencer dan engagement
df["Social_Media_Influence"] = df["Social_Media_Influence"].fillna("Unknown")
df["Engagement_with_Ads"]    = df["Engagement_with_Ads"].fillna("Unknown")
print("Missing values setelah penanganan:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("→ Tidak ada missing values tersisa.")

Missing values setelah penanganan:
Series([], dtype: int64)
→ Tidak ada missing values tersisa.


In [12]:
#Ubah nama collumn agar lebih rapih
df.rename(columns={
    "Time_Spent_on_Product_Research(hours)": "Research_Time_Hours",
    "Customer_Loyalty_Program_Member"       : "Loyalty_Member",
}, inplace=True)
print("Kolom setelah rename:")
print(df.columns.tolist())

Kolom setelah rename:
['Customer_ID', 'Age', 'Gender', 'Marital_Status', 'Education_Level', 'Income_Class', 'Location', 'Purchase_Category', 'Purchase_Amount', 'Frequency_of_Purchase', 'Purchase_Channel', 'Brand_Loyalty', 'Product_Rating', 'Research_Time_Hours', 'Social_Media_Influence', 'Discount_Sensitivity', 'Return_Rate', 'Customer_Satisfaction', 'Engagement_with_Ads', 'Device_Used_for_Shopping', 'Payment_Method', 'Time_of_Purchase', 'Discount_Used', 'Loyalty_Member', 'Purchase_Intent', 'Shipping_Preference', 'Time_to_Decision', 'Purchase_Month', 'Purchase_Month_Name', 'Purchase_Year', 'Purchase_DayOfWeek', 'Purchase_Quarter']


In [13]:
#Hasil akhir
print(f"Shape akhir      : {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Missing values   : {df.isnull().sum().sum()}")
print(f"Duplicate rows   : {df.duplicated().sum()}")
print(f"\nTipe data per kolom:\n{df.dtypes}")
print(f"\nSample 3 baris pertama:")
print(df.head(3).to_string())

Shape akhir      : 1000 baris, 32 kolom
Missing values   : 0
Duplicate rows   : 0

Tipe data per kolom:
Customer_ID                         object
Age                                  int64
Gender                              object
Marital_Status                      object
Education_Level                     object
Income_Class                        object
Location                            object
Purchase_Category                   object
Purchase_Amount                    float64
Frequency_of_Purchase                int64
Purchase_Channel                    object
Brand_Loyalty                        int64
Product_Rating                       int64
Research_Time_Hours                float64
Social_Media_Influence              object
Discount_Sensitivity                object
Return_Rate                          int64
Customer_Satisfaction                int64
Engagement_with_Ads                 object
Device_Used_for_Shopping            object
Payment_Method                      

In [14]:
from pathlib import Path
DATA = Path.cwd().parent / "data"

#Simpan data
output_path = DATA / "ecommerce_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Data bersih berhasil disimpan ke: {output_path}")

Data bersih berhasil disimpan ke: <repo>\09-ecommerce-analytics\data\ecommerce_cleaned.csv
